In [1]:
import os
import numpy as np
import pandas as pd



In [2]:
sub_path = "../input/ventilator-pressure-prediction/sample_submission.csv"
sub = pd.read_csv(sub_path)

paths = {
    "sub_1": "../input/improvement-base-on-tensor-bidirect-lstm-0-173/submission.csv",
    "sub_2": "../input/finetune-of-tensorflow-bidirectional-lstm/submission.csv",
    "sub_3": "../input/lightautoml-bidirectional-lstm/submission.csv",
    "sub_4": "../input/tensorflow-bidirectional-lstm-custom-mae-loss/submission.csv",
}

loaded = {}
missing = []
for name, p in paths.items():
    if os.path.exists(p):
        df = pd.read_csv(p)
        if "id" not in df.columns or "pressure" not in df.columns:
            missing.append(name)
        else:
            loaded[name] = df
    else:
        missing.append(name)

use_ensemble = len(missing) == 0



In [3]:
if use_ensemble:
    sub_1, sub_2, sub_3, sub_4 = (
        loaded["sub_1"],
        loaded["sub_2"],
        loaded["sub_3"],
        loaded["sub_4"],
    )

    def align(df):
        return df.sort_values("id").reset_index(drop=True)

    sub_sorted = align(sub)
    p1 = align(sub_1)["pressure"].to_numpy()
    p2 = align(sub_2)["pressure"].to_numpy()
    p3 = align(sub_3)["pressure"].to_numpy()
    p4 = align(sub_4)["pressure"].to_numpy()

    sub_sorted["pressure"] = (p1 * 0.1) + (p2 * 0.6) + (p3 * 0.2) + (p4 * 0.1)
    sub = sub_sorted
else:
    train_path = "../input/ventilator-pressure-prediction/train.csv"
    test_path = "../input/ventilator-pressure-prediction/test.csv"
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    train_all = train.loc[
        :,
        ["R", "C", "breath_id", "time_step", "u_in", "u_out", "pressure"],
    ].copy()

    def add_step(df: pd.DataFrame) -> pd.DataFrame:
        # Keep core logic identical; step is still within-breath row index.
        df = df.sort_values(["breath_id", "time_step"]).copy()
        df["step"] = df.groupby("breath_id", sort=False).cumcount().astype(np.int16)

        df["u_in_r"] = df["u_in"].round(2).astype(np.float32)
        df["u_in_r1"] = df["u_in"].round(1).astype(np.float32)
        return df

    train_all = add_step(train_all)
    test = add_step(test)

    def add_uin_cum_dt(df: pd.DataFrame) -> pd.DataFrame:
        df = df.sort_values(["breath_id", "step"]).copy()

        dt_raw = (
            df.groupby("breath_id", sort=False)["time_step"].diff().astype(np.float32)
        )

        pos = dt_raw.where(dt_raw > 0)
        dt_min_pos = (
            pos.groupby(df["breath_id"], sort=False).transform("min").astype(np.float32)
        )
        dt_min_pos = dt_min_pos.fillna(np.float32(0.03))

        dt = dt_raw.fillna(dt_min_pos).clip(lower=0.0).astype(np.float32)
        df["dt"] = dt

        u_out = df["u_out"].astype(np.float32)
        insp_gate = (1.0 - u_out).astype(np.float32)
        u_in_dt = (df["u_in"].astype(np.float32) * insp_gate * df["dt"]).astype(
            np.float32
        )

        df["u_in_cum_dt"] = (
            u_in_dt.groupby(df["breath_id"], sort=False).cumsum().astype(np.float32)
        )

        # Score-relevant minimal change:
        # Make cumulative discretizations consistent and usable for joins by prioritizing coarse rounding.
        # (Fine rounding causes near-total key mismatch between train/test; coarse-first reduces fallback rate.)
        df["u_in_cum_dt_r1"] = df["u_in_cum_dt"].round(1).astype(np.float32)
        df["u_in_cum_dt_r2"] = df["u_in_cum_dt"].round(2).astype(np.float32)
        df["u_in_cum_dt_r3"] = df["u_in_cum_dt"].round(3).astype(np.float32)
        df["u_in_cum_dt_r"] = df["u_in_cum_dt"].round(4).astype(np.float32)

        return df

    train_all = add_uin_cum_dt(train_all)
    test = add_uin_cum_dt(test)

    # Build lookup tables (same approach: medians over hierarchical keys)
    key_cum_full1 = ["R", "C", "step", "u_in_cum_dt_r1", "u_out"]
    med_cum_full1 = (
        train_all.groupby(key_cum_full1, observed=True)["pressure"]
        .median()
        .rename("pred_cum_full1")
        .reset_index()
    )

    key_cum_su1 = ["R", "C", "step", "u_in_cum_dt_r1"]
    med_cum_su1 = (
        train_all.groupby(key_cum_su1, observed=True)["pressure"]
        .median()
        .rename("pred_cum_su1")
        .reset_index()
    )

    key_cum_full2 = ["R", "C", "step", "u_in_cum_dt_r2", "u_out"]
    med_cum_full2 = (
        train_all.groupby(key_cum_full2, observed=True)["pressure"]
        .median()
        .rename("pred_cum_full2")
        .reset_index()
    )

    key_cum_su2 = ["R", "C", "step", "u_in_cum_dt_r2"]
    med_cum_su2 = (
        train_all.groupby(key_cum_su2, observed=True)["pressure"]
        .median()
        .rename("pred_cum_su2")
        .reset_index()
    )

    key_cum_full3 = ["R", "C", "step", "u_in_cum_dt_r3", "u_out"]
    med_cum_full3 = (
        train_all.groupby(key_cum_full3, observed=True)["pressure"]
        .median()
        .rename("pred_cum_full3")
        .reset_index()
    )

    key_cum_su3 = ["R", "C", "step", "u_in_cum_dt_r3"]
    med_cum_su3 = (
        train_all.groupby(key_cum_su3, observed=True)["pressure"]
        .median()
        .rename("pred_cum_su3")
        .reset_index()
    )

    key_cum_full = ["R", "C", "step", "u_in_cum_dt_r", "u_out"]
    med_cum_full = (
        train_all.groupby(key_cum_full, observed=True)["pressure"]
        .median()
        .rename("pred_cum_full")
        .reset_index()
    )

    key_cum_su = ["R", "C", "step", "u_in_cum_dt_r"]
    med_cum_su = (
        train_all.groupby(key_cum_su, observed=True)["pressure"]
        .median()
        .rename("pred_cum_su")
        .reset_index()
    )

    key_full = ["R", "C", "step", "u_in_r", "u_out"]
    med_full = (
        train_all.groupby(key_full, observed=True)["pressure"]
        .median()
        .rename("pred_full")
        .reset_index()
    )

    test2 = test[
        [
            "id",
            "R",
            "C",
            "step",
            "u_in_r",
            "u_in_r1",
            "u_in_cum_dt_r",
            "u_in_cum_dt_r3",
            "u_in_cum_dt_r2",
            "u_in_cum_dt_r1",
            "u_out",
            "breath_id",
            "time_step",
            "u_in",
        ]
    ].copy()

    # Merges (same tables, just keep all available for fill hierarchy)
    test2 = test2.merge(med_cum_full1, on=key_cum_full1, how="left")
    test2 = test2.merge(med_cum_su1, on=key_cum_su1, how="left")
    test2 = test2.merge(med_cum_full2, on=key_cum_full2, how="left")
    test2 = test2.merge(med_cum_su2, on=key_cum_su2, how="left")
    test2 = test2.merge(med_cum_full3, on=key_cum_full3, how="left")
    test2 = test2.merge(med_cum_su3, on=key_cum_su3, how="left")
    test2 = test2.merge(med_cum_full, on=key_cum_full, how="left")
    test2 = test2.merge(med_cum_su, on=key_cum_su, how="left")
    test2 = test2.merge(med_full, on=key_full, how="left")

    key_step = ["R", "C", "step", "u_out"]
    med_step = (
        train_all.groupby(key_step, observed=True)["pressure"]
        .median()
        .rename("pred_step")
        .reset_index()
    )
    test2 = test2.merge(med_step, on=key_step, how="left")

    key_su = ["R", "C", "step", "u_in_r"]
    med_su = (
        train_all.groupby(key_su, observed=True)["pressure"]
        .median()
        .rename("pred_su")
        .reset_index()
    )
    test2 = test2.merge(med_su, on=key_su, how="left")

    key_su1o = ["R", "C", "step", "u_in_r1", "u_out"]
    med_su1o = (
        train_all.groupby(key_su1o, observed=True)["pressure"]
        .median()
        .rename("pred_su1o")
        .reset_index()
    )
    test2 = test2.merge(med_su1o, on=key_su1o, how="left")

    key_u = ["R", "C", "u_in_r", "u_out"]
    med_u = (
        train_all.groupby(key_u, observed=True)["pressure"]
        .median()
        .rename("pred_u")
        .reset_index()
    )
    test2 = test2.merge(med_u, on=key_u, how="left")

    key_u1 = ["R", "C", "u_in_r1", "u_out"]
    med_u1 = (
        train_all.groupby(key_u1, observed=True)["pressure"]
        .median()
        .rename("pred_u1")
        .reset_index()
    )
    test2 = test2.merge(med_u1, on=key_u1, how="left")

    key_rc = ["R", "C", "u_out"]
    med_rc = (
        train_all.groupby(key_rc, observed=True)["pressure"]
        .median()
        .rename("pred_rc")
        .reset_index()
    )
    test2 = test2.merge(med_rc, on=key_rc, how="left")

    global_med = float(train_all["pressure"].median())

    # Score-relevant minimal change:
    # Use coarse cumulative keys first (r1 -> r2), then finer (r3 -> r4).
    # This reduces missing-key fallbacks drastically while preserving identical median-lookup semantics.
    pred = test2["pred_cum_full1"]
    pred = pred.fillna(test2["pred_cum_su1"])
    pred = pred.fillna(test2["pred_cum_full2"])
    pred = pred.fillna(test2["pred_cum_su2"])
    pred = pred.fillna(test2["pred_cum_full3"])
    pred = pred.fillna(test2["pred_cum_su3"])
    pred = pred.fillna(test2["pred_cum_full"])
    pred = pred.fillna(test2["pred_cum_su"])
    pred = pred.fillna(test2["pred_full"])
    pred = pred.fillna(test2["pred_step"])
    pred = pred.fillna(test2["pred_su"])
    pred = pred.fillna(test2["pred_su1o"])
    pred = pred.fillna(test2["pred_u"])
    pred = pred.fillna(test2["pred_u1"])
    pred = pred.fillna(test2["pred_rc"])
    pred = pred.fillna(global_med)

    pressure_values = np.sort(train["pressure"].unique()).astype(np.float32)

    def snap_to_grid(x: np.ndarray, grid: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32, copy=False)
        idx = np.searchsorted(grid, x, side="left")
        idx = np.clip(idx, 0, len(grid) - 1)
        left = np.clip(idx - 1, 0, len(grid) - 1)
        right = idx
        choose_right = np.abs(grid[right] - x) <= np.abs(x - grid[left])
        snapped = np.where(choose_right, grid[right], grid[left]).astype(np.float32)
        return snapped

    pred_np = pred.to_numpy(dtype=np.float32)
    pred_np = snap_to_grid(pred_np, pressure_values)

    sub = sub.sort_values("id").reset_index(drop=True)
    test2 = test2.sort_values("id").reset_index(drop=True)
    sub["pressure"] = pred_np

sub[["id", "pressure"]].to_csv("submission.csv", index=False)

sub.head(5)

/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)


,id,pressure
0,1,6.118701
1,2,7.032628
2,3,8.719880
3,4,10.125922
4,5,10.125922
